In [1]:
# 🗺️ Interactive Mumbai Locality Map

This notebook creates an interactive property intelligence map.

Objectives:

- Visualize properties on an interactive map
- Display locality-wise property information
- Analyze average property price
- Analyze price per sq.ft
- Analyze predicted price
- Show property locations using latitude and longitude
- Create locality-level markers
- Add interactive popups
- Create price-based visualization
- Save the interactive map as an HTML file
- Prepare the map for Flask dashboard integration

SyntaxError: invalid syntax (3832428122.py, line 3)

In [2]:
import os
import numpy as np
import pandas as pd
import folium

from folium.plugins import MarkerCluster

from IPython.display import display, IFrame

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
PROJECT_DIR = r"C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction"

PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "processed"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Project Directory:")
print(PROJECT_DIR)

print("\nOutput Directory:")
print(OUTPUT_DIR)

Project Directory:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction

Output Directory:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs


In [4]:
print("Searching for property_intelligence.csv...")
print()

found_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if file.lower() == "property_intelligence.csv":
            found_files.append(
                os.path.join(root, file)
            )

if len(found_files) == 0:
    
    print("❌ property_intelligence.csv was NOT found.")
    print()
    print("Project folder checked:")
    print(PROJECT_DIR)

else:
    
    print("✅ File found!")
    print()
    
    for i, file_path in enumerate(found_files):
        print(
            f"{i + 1}. {file_path}"
        )
    
    property_path = found_files[0]
    
    print()
    print("Selected file:")
    print(property_path)

Searching for property_intelligence.csv...

✅ File found!

1. C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\data\processed\property_intelligence.csv

Selected file:
C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\data\processed\property_intelligence.csv


In [5]:
df = pd.read_csv(
    property_path
)

print("Dataset loaded successfully!")

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!

Shape:
(51767, 22)

Columns:
['price', 'area', 'price_per_sqft', 'locality', 'city', 'property_type', 'bedroom_num', 'bathroom_num', 'balcony_num', 'furnished', 'age', 'total_floors', 'latitude', 'longitude', 'predicted_price', 'price_difference', 'price_difference_percent', 'predicted_price_per_sqft', 'price_per_sqft_difference', 'anomaly_label', 'anomaly_score', 'is_anomaly']


In [6]:
display(
    df.head()
)

,price,area,price_per_sqft,locality,city,property_type,bedroom_num,bathroom_num,balcony_num,furnished,...,latitude,longitude,predicted_price,price_difference,price_difference_percent,predicted_price_per_sqft,price_per_sqft_difference,anomaly_label,anomaly_score,is_anomaly
0,6600283,757,8719.000000,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,...,19.244410,73.123253,5.749573e+06,8.507096e+05,14.796047,7595.209299,1123.790701,1,0.261409,False
1,6169841,652,9462.946319,Kalyan,Mumbai,Apartment,2,2,0,Unfurnished,...,19.257294,73.148872,6.422073e+06,-2.522321e+05,-3.927581,9849.805444,-386.859125,1,0.257219,False
2,4599936,396,11616.000000,Dombivali,Mumbai,Apartment,1,1,0,Unfurnished,...,19.209026,73.081276,4.378319e+06,2.216174e+05,5.061702,11056.360025,559.639975,1,0.242774,False
3,51980000,1130,46000.000000,Ville Parle,Mumbai,Apartment,3,3,0,Unfurnished,...,19.097841,72.851158,5.084753e+07,1.132471e+06,2.227190,44997.813310,1002.186690,1,0.214435,False
4,3915000,435,9000.000000,Nala Sopara,Mumbai,Apartment,1,1,0,Unfurnished,...,19.420601,72.809319,3.663747e+06,2.512535e+05,6.857829,8422.405793,577.594207,1,0.208075,False


In [7]:
required_columns = [
    "locality",
    "latitude",
    "longitude",
    "price",
    "area",
    "price_per_sqft",
    "predicted_price",
    "predicted_price_per_sqft",
    "property_type",
    "bedroom_num",
    "bathroom_num"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if len(missing_columns) == 0:
    
    print(
        "✅ All required map columns are available."
    )

else:
    
    print(
        "❌ Missing columns:"
    )
    
    print(
        missing_columns
    )

✅ All required map columns are available.


In [8]:
print("Latitude Statistics")
print("=" * 60)

display(
    df["latitude"].describe()
)

print("\nLongitude Statistics")
print("=" * 60)

display(
    df["longitude"].describe()
)

Latitude Statistics


count    51767.000000
mean        19.172944
std          0.532951
min         12.889899
25%         19.069774
50%         19.160484
75%         19.228041
max         72.868339
Name: latitude, dtype: float64


Longitude Statistics


count    51767.000000
mean        72.966697
std          0.457737
min         72.435379
25%         72.856712
50%         72.913185
75%         73.021202
max         91.804413
Name: longitude, dtype: float64

In [9]:
missing_coordinates = df[
    ["latitude", "longitude"]
].isnull().sum()

print(
    "Missing Coordinate Values:"
)

display(
    missing_coordinates
)

Missing Coordinate Values:


latitude     0
longitude    0
dtype: int64

In [10]:
map_df = df[
    required_columns
].copy()

map_df = map_df.dropna(
    subset=[
        "latitude",
        "longitude",
        "locality"
    ]
)

map_df["latitude"] = pd.to_numeric(
    map_df["latitude"],
    errors="coerce"
)

map_df["longitude"] = pd.to_numeric(
    map_df["longitude"],
    errors="coerce"
)

map_df = map_df.dropna(
    subset=[
        "latitude",
        "longitude"
    ]
)

print(
    "Clean map dataset created."
)

print(
    "Shape:",
    map_df.shape
)

Clean map dataset created.
Shape: (51767, 11)


In [11]:
map_df = map_df[
    (map_df["latitude"] >= 18.0)
    &
    (map_df["latitude"] <= 20.5)
    &
    (map_df["longitude"] >= 72.0)
    &
    (map_df["longitude"] <= 73.5)
].copy()

print(
    "Valid geographic records:"
)

print(
    len(map_df)
)

Valid geographic records:
51613


In [12]:
print("🗺️ MAP DATASET SUMMARY")
print("=" * 70)

print(
    f"Total properties: {len(map_df):,}"
)

print(
    f"Unique localities: "
    f"{map_df['locality'].nunique():,}"
)

print(
    f"Latitude range: "
    f"{map_df['latitude'].min():.4f} "
    f"to "
    f"{map_df['latitude'].max():.4f}"
)

print(
    f"Longitude range: "
    f"{map_df['longitude'].min():.4f} "
    f"to "
    f"{map_df['longitude'].max():.4f}"
)

🗺️ MAP DATASET SUMMARY
Total properties: 51,613
Unique localities: 406
Latitude range: 18.7494 to 19.9826
Longitude range: 72.7084 to 73.4943


In [13]:
mumbai_map = folium.Map(
    location=[
        19.0760,
        72.8777
    ],
    zoom_start=11,
    tiles="OpenStreetMap"
)

print(
    "Mumbai base map created successfully!"
)

mumbai_map

Mumbai base map created successfully!


In [14]:
property_cluster = MarkerCluster(
    name="Properties"
)

property_cluster.add_to(
    mumbai_map
)

print(
    "Property marker cluster created."
)

Property marker cluster created.


In [15]:
sample_size = min(
    2000,
    len(map_df)
)

map_sample = map_df.sample(
    n=sample_size,
    random_state=42
)

print(
    f"Adding {sample_size:,} properties to map..."
)

for _, row in map_sample.iterrows():
    
    popup_text = f"""
    <b>Locality:</b> {row['locality']}<br>
    <b>Property Type:</b> {row['property_type']}<br>
    <b>Area:</b> {row['area']:,.0f} sq.ft<br>
    <b>Price:</b> ₹{row['price']:,.0f}<br>
    <b>Price/sq.ft:</b> ₹{row['price_per_sqft']:,.0f}<br>
    <b>Predicted Price:</b> ₹{row['predicted_price']:,.0f}<br>
    <b>Predicted Price/sq.ft:</b> ₹{row['predicted_price_per_sqft']:,.0f}<br>
    <b>Bedrooms:</b> {row['bedroom_num']}<br>
    <b>Bathrooms:</b> {row['bathroom_num']}
    """
    
    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=4,
        popup=folium.Popup(
            popup_text,
            max_width=350
        ),
        fill=True
    ).add_to(
        property_cluster
    )

print(
    "✅ Property markers added successfully!"
)

Adding 2,000 properties to map...
✅ Property markers added successfully!


In [16]:
locality_summary = (
    map_df
    .groupby("locality")
    .agg(
        property_count=(
            "price",
            "count"
        ),
        average_price=(
            "price",
            "mean"
        ),
        median_price=(
            "price",
            "median"
        ),
        average_price_per_sqft=(
            "price_per_sqft",
            "mean"
        ),
        average_predicted_price=(
            "predicted_price",
            "mean"
        ),
        average_predicted_price_per_sqft=(
            "predicted_price_per_sqft",
            "mean"
        ),
        average_latitude=(
            "latitude",
            "mean"
        ),
        average_longitude=(
            "longitude",
            "mean"
        )
    )
    .reset_index()
)

display(
    locality_summary.head(20)
)

,locality,property_count,average_price,median_price,average_price_per_sqft,average_predicted_price,average_predicted_price_per_sqft,average_latitude,average_longitude
0,Acc Cement Road,1,1.000000e+07,10000000.0,15037.593985,9.380265e+06,14105.661627,19.180456,72.958420
1,Adarsh Nagar,2,1.060938e+08,106093750.0,72716.346154,9.885211e+07,67756.829785,19.014942,72.821365
2,Additional M.I.D.C,2,3.400000e+06,3400000.0,4236.955187,3.374729e+06,4178.170401,19.187177,73.191494
3,Adharwadi,2,3.173000e+06,3173000.0,7277.669903,3.223261e+06,7256.768294,19.252434,73.125969
4,Agasan Village,2,3.085000e+07,30850000.0,31148.284314,2.618037e+07,26489.172477,19.046006,72.943802
5,Agripada,40,3.854632e+07,28000000.0,28420.674294,4.100053e+07,30368.456608,19.026414,72.840731
6,Airoli,240,1.341590e+07,13000000.0,13772.108755,1.362530e+07,13880.010541,19.154944,72.994974
7,Altamount Road,1,2.100000e+08,210000000.0,77777.777778,2.341971e+08,86739.681300,18.965179,72.809486
8,Ambarnath,27,2.983333e+06,2800000.0,4193.952931,3.064536e+06,4274.998734,19.204630,73.196250
9,Ambernath,975,3.380517e+06,3100000.0,4676.487401,3.401939e+06,4699.771404,19.197422,73.190202


In [17]:
print(
    "Locality Summary Shape:"
)

print(
    locality_summary.shape
)

print(
    "\nNumber of localities:"
)

print(
    locality_summary["locality"].nunique()
)

Locality Summary Shape:
(406, 9)

Number of localities:
406


In [18]:
top_localities = (
    locality_summary
    .sort_values(
        "property_count",
        ascending=False
    )
    .head(20)
)

display(
    top_localities
)

,locality,property_count,average_price,median_price,average_price_per_sqft,average_predicted_price,average_predicted_price_per_sqft,average_latitude,average_longitude
361,Thane,5284,1.577249e+07,12600000.0,15617.377475,1.577969e+07,15638.471188,19.232078,72.976177
207,Mira Road,3157,8.740131e+06,7900000.0,11341.205611,8.810858e+06,11387.432035,19.283842,72.876008
140,Kandivali,2030,1.901912e+07,16000000.0,21156.776353,1.904568e+07,21142.885966,19.205548,72.852828
15,Andheri,2026,3.225508e+07,22831785.0,27326.998357,3.292498e+07,27657.686386,19.127179,72.848674
158,Kharghar,1900,1.396480e+07,12200000.0,11434.054346,1.395762e+07,11450.396391,19.064797,73.073162
85,Dombivali,1869,6.022477e+06,5390920.0,8239.683142,6.049791e+06,8272.566215,19.197334,73.092100
136,Kalyan,1736,6.491410e+06,5275000.0,7664.293157,6.505430e+06,7681.624119,19.245053,73.131187
46,Borivali,1675,2.100590e+07,17463690.0,23541.602934,2.106759e+07,23615.740624,19.228486,72.854966
58,Chembur,1586,2.378321e+07,18899999.5,23967.566029,2.390857e+07,24021.788956,19.055061,72.897570
191,Malad,1518,1.941179e+07,16917015.0,20967.442385,1.969340e+07,21118.646739,19.186788,72.840521


In [20]:
locality_map = folium.Map(
    location=[
        19.0760,
        72.8777
    ],
    zoom_start=11,
    tiles="OpenStreetMap"
)

print(
    "Locality map created."
)

Locality map created.


In [21]:
locality_cluster = MarkerCluster(
    name="Localities"
)

locality_cluster.add_to(
    locality_map
)

print(
    "Locality marker cluster created."
)

Locality marker cluster created.


In [22]:
for _, row in locality_summary.iterrows():
    
    popup_text = f"""
    <div style="width:300px">
    
    <h4>{row['locality']}</h4>
    
    <b>Properties:</b>
    {row['property_count']:,}
    <br><br>
    
    <b>Average Price:</b>
    ₹{row['average_price']:,.0f}
    <br>
    
    <b>Median Price:</b>
    ₹{row['median_price']:,.0f}
    <br>
    
    <b>Average Price/Sq.Ft:</b>
    ₹{row['average_price_per_sqft']:,.0f}
    <br>
    
    <b>Average Predicted Price:</b>
    ₹{row['average_predicted_price']:,.0f}
    <br>
    
    <b>Predicted Price/Sq.Ft:</b>
    ₹{row['average_predicted_price_per_sqft']:,.0f}
    
    </div>
    """
    
    folium.Marker(
        location=[
            row["average_latitude"],
            row["average_longitude"]
        ],
        popup=folium.Popup(
            popup_text,
            max_width=400
        ),
        tooltip=str(
            row["locality"]
        )
    ).add_to(
        locality_cluster
    )

print(
    "✅ Locality markers added!"
)

✅ Locality markers added!


In [23]:
ppsf_map = folium.Map(
    location=[
        19.0760,
        72.8777
    ],
    zoom_start=11,
    tiles="OpenStreetMap"
)

print(
    "Price-per-sq.ft map created."
)

Price-per-sq.ft map created.


In [24]:
def classify_ppsf(value):
    
    if value < 10000:
        return "Lower Price"
    
    elif value < 20000:
        return "Medium Price"
    
    elif value < 30000:
        return "High Price"
    
    else:
        return "Very High Price"


locality_summary["ppsf_category"] = (
    locality_summary[
        "average_price_per_sqft"
    ]
    .apply(classify_ppsf)
)

display(
    locality_summary[
        [
            "locality",
            "average_price_per_sqft",
            "ppsf_category"
        ]
    ].head(20)
)

,locality,average_price_per_sqft,ppsf_category
0,Acc Cement Road,15037.593985,Medium Price
1,Adarsh Nagar,72716.346154,Very High Price
2,Additional M.I.D.C,4236.955187,Lower Price
3,Adharwadi,7277.669903,Lower Price
4,Agasan Village,31148.284314,Very High Price
5,Agripada,28420.674294,High Price
6,Airoli,13772.108755,Medium Price
7,Altamount Road,77777.777778,Very High Price
8,Ambarnath,4193.952931,Lower Price
9,Ambernath,4676.487401,Lower Price


In [25]:
price_cluster = MarkerCluster(
    name="Price Per Sq.Ft"
)

price_cluster.add_to(
    ppsf_map
)

for _, row in locality_summary.iterrows():
    
    popup_text = f"""
    <b>Locality:</b> {row['locality']}<br>
    <b>Average Price/Sq.Ft:</b>
    ₹{row['average_price_per_sqft']:,.0f}<br>
    <b>Category:</b>
    {row['ppsf_category']}<br>
    <b>Properties:</b>
    {row['property_count']:,}
    """
    
    folium.CircleMarker(
        location=[
            row["average_latitude"],
            row["average_longitude"]
        ],
        radius=7,
        popup=folium.Popup(
            popup_text,
            max_width=300
        ),
        tooltip=(
            f"{row['locality']} | "
            f"₹{row['average_price_per_sqft']:,.0f}/sq.ft"
        ),
        fill=True
    ).add_to(
        price_cluster
    )

print(
    "✅ Price markers added!"
)

✅ Price markers added!


In [26]:
folium.LayerControl().add_to(
    ppsf_map
)

print(
    "Layer control added successfully!"
)

Layer control added successfully!


In [27]:
ppsf_map

In [28]:
final_map = folium.Map(
    location=[
        19.0760,
        72.8777
    ],
    zoom_start=11,
    tiles="OpenStreetMap"
)

print(
    "Final interactive map created."
)

Final interactive map created.


In [29]:
property_layer = folium.FeatureGroup(
    name="Property Locations"
)

property_layer.add_to(
    final_map
)

for _, row in map_sample.iterrows():
    
    popup_text = f"""
    <b>Locality:</b> {row['locality']}<br>
    <b>Property Type:</b> {row['property_type']}<br>
    <b>Area:</b> {row['area']:,.0f} sq.ft<br>
    <b>Price:</b> ₹{row['price']:,.0f}<br>
    <b>Actual ₹/sq.ft:</b>
    ₹{row['price_per_sqft']:,.0f}<br>
    <b>Predicted Price:</b>
    ₹{row['predicted_price']:,.0f}<br>
    <b>Predicted ₹/sq.ft:</b>
    ₹{row['predicted_price_per_sqft']:,.0f}
    """
    
    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=3,
        popup=folium.Popup(
            popup_text,
            max_width=350
        ),
        fill=True
    ).add_to(
        property_layer
    )

print(
    "Property layer added."
)

Property layer added.


In [30]:
locality_layer = folium.FeatureGroup(
    name="Locality Intelligence"
)

locality_layer.add_to(
    final_map
)

for _, row in locality_summary.iterrows():
    
    popup_text = f"""
    <div style="width:300px">
    
    <h4>{row['locality']}</h4>
    
    <b>Listings:</b>
    {row['property_count']:,}<br>
    
    <b>Average Price:</b>
    ₹{row['average_price']:,.0f}<br>
    
    <b>Median Price:</b>
    ₹{row['median_price']:,.0f}<br>
    
    <b>Average ₹/Sq.Ft:</b>
    ₹{row['average_price_per_sqft']:,.0f}<br>
    
    <b>Predicted ₹/Sq.Ft:</b>
    ₹{row['average_predicted_price_per_sqft']:,.0f}
    
    </div>
    """
    
    folium.Marker(
        location=[
            row["average_latitude"],
            row["average_longitude"]
        ],
        popup=folium.Popup(
            popup_text,
            max_width=400
        ),
        tooltip=str(
            row["locality"]
        )
    ).add_to(
        locality_layer
    )

print(
    "Locality intelligence layer added."
)

Locality intelligence layer added.


In [31]:
folium.LayerControl().add_to(
    final_map
)

print(
    "Layer control added."
)

Layer control added.


In [32]:
final_map

In [33]:
main_map_path = os.path.join(
    OUTPUT_DIR,
    "mumbai_interactive_property_map.html"
)

final_map.save(
    main_map_path
)

print(
    "✅ Main interactive map saved!"
)

print(
    "Path:",
    main_map_path
)

✅ Main interactive map saved!
Path: C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs\mumbai_interactive_property_map.html


In [34]:
locality_map_path = os.path.join(
    OUTPUT_DIR,
    "mumbai_locality_map.html"
)

locality_map.save(
    locality_map_path
)

print(
    "✅ Locality map saved!"
)

print(
    "Path:",
    locality_map_path
)

✅ Locality map saved!
Path: C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs\mumbai_locality_map.html


In [35]:
ppsf_map_path = os.path.join(
    OUTPUT_DIR,
    "mumbai_price_per_sqft_map.html"
)

ppsf_map.save(
    ppsf_map_path
)

print(
    "✅ Price-per-sq.ft map saved!"
)

print(
    "Path:",
    ppsf_map_path
)

✅ Price-per-sq.ft map saved!
Path: C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs\mumbai_price_per_sqft_map.html


In [36]:
locality_summary_path = os.path.join(
    OUTPUT_DIR,
    "interactive_locality_summary.csv"
)

locality_summary.to_csv(
    locality_summary_path,
    index=False
)

print(
    "✅ Locality summary saved!"
)

print(
    "Path:",
    locality_summary_path
)

✅ Locality summary saved!
Path: C:\Users\Tejas Dalvi\OneDrive\Documents\mumbai-house-price-prediction\outputs\interactive_locality_summary.csv


In [37]:
output_files = [
    main_map_path,
    locality_map_path,
    ppsf_map_path,
    locality_summary_path
]

print(
    "📁 OUTPUT FILE VERIFICATION"
)

print("=" * 70)

for file_path in output_files:
    
    if os.path.exists(file_path):
        
        size_kb = (
            os.path.getsize(file_path)
            / 1024
        )
        
        print(
            f"✅ {os.path.basename(file_path)} "
            f"({size_kb:.1f} KB)"
        )
        
    else:
        
        print(
            f"❌ {os.path.basename(file_path)}"
        )

📁 OUTPUT FILE VERIFICATION
✅ mumbai_interactive_property_map.html (3075.5 KB)
✅ mumbai_locality_map.html (540.7 KB)
✅ mumbai_price_per_sqft_map.html (574.2 KB)
✅ interactive_locality_summary.csv (53.2 KB)


In [38]:
IFrame(
    src=main_map_path,
    width="100%",
    height=600
)

In [39]:
top_localities_summary = (
    locality_summary
    .sort_values(
        "property_count",
        ascending=False
    )
    .head(20)
)

display(
    top_localities_summary[
        [
            "locality",
            "property_count",
            "average_price",
            "average_price_per_sqft",
            "average_predicted_price",
            "average_predicted_price_per_sqft",
            "ppsf_category"
        ]
    ]
)

,locality,property_count,average_price,average_price_per_sqft,average_predicted_price,average_predicted_price_per_sqft,ppsf_category
361,Thane,5284,1.577249e+07,15617.377475,1.577969e+07,15638.471188,Medium Price
207,Mira Road,3157,8.740131e+06,11341.205611,8.810858e+06,11387.432035,Medium Price
140,Kandivali,2030,1.901912e+07,21156.776353,1.904568e+07,21142.885966,High Price
15,Andheri,2026,3.225508e+07,27326.998357,3.292498e+07,27657.686386,High Price
158,Kharghar,1900,1.396480e+07,11434.054346,1.395762e+07,11450.396391,Medium Price
85,Dombivali,1869,6.022477e+06,8239.683142,6.049791e+06,8272.566215,Lower Price
136,Kalyan,1736,6.491410e+06,7664.293157,6.505430e+06,7681.624119,Lower Price
46,Borivali,1675,2.100590e+07,23541.602934,2.106759e+07,23615.740624,High Price
58,Chembur,1586,2.378321e+07,23967.566029,2.390857e+07,24021.788956,High Price
191,Malad,1518,1.941179e+07,20967.442385,1.969340e+07,21118.646739,High Price


In [40]:
most_expensive_localities = (
    locality_summary
    .sort_values(
        "average_price_per_sqft",
        ascending=False
    )
    .head(15)
)

print(
    "💰 MOST EXPENSIVE LOCALITIES BY PRICE/SQ.FT"
)

print("=" * 70)

display(
    most_expensive_localities[
        [
            "locality",
            "property_count",
            "average_price_per_sqft",
            "average_predicted_price_per_sqft",
            "ppsf_category"
        ]
    ]
)

💰 MOST EXPENSIVE LOCALITIES BY PRICE/SQ.FT


,locality,property_count,average_price_per_sqft,average_predicted_price_per_sqft,ppsf_category
226,Napean Sea Road,1,132375.189673,115892.847271,Very High Price
199,Marine Drive,2,114175.602818,85488.554220,Very High Price
65,Churchgate,5,98918.281466,95290.818231,Very High Price
30,Bandstand,1,96296.296296,79511.111111,Very High Price
254,Pali Hill,3,90804.799648,79040.813396,Very High Price
7,Altamount Road,1,77777.777778,86739.681300,Very High Price
48,Breach Candy,2,77619.047619,66855.267062,Very High Price
190,Malabar Hill,36,76152.100253,75369.578876,Very High Price
1,Adarsh Nagar,2,72716.346154,67756.829785,Very High Price
93,Gamdevi,3,69457.117715,64473.684270,Very High Price


In [41]:
print("🗺️ INTERACTIVE MAP SUMMARY")
print("=" * 70)

print(
    f"Properties used for map: "
    f"{len(map_sample):,}"
)

print(
    f"Total valid geographic properties: "
    f"{len(map_df):,}"
)

print(
    f"Unique localities: "
    f"{locality_summary['locality'].nunique():,}"
)

print(
    f"Average property price: "
    f"₹{map_df['price'].mean():,.0f}"
)

print(
    f"Average price/sq.ft: "
    f"₹{map_df['price_per_sqft'].mean():,.0f}"
)

print(
    f"Average predicted price: "
    f"₹{map_df['predicted_price'].mean():,.0f}"
)

print(
    f"Average predicted price/sq.ft: "
    f"₹{map_df['predicted_price_per_sqft'].mean():,.0f}"
)

🗺️ INTERACTIVE MAP SUMMARY
Properties used for map: 2,000
Total valid geographic properties: 51,613
Unique localities: 406
Average property price: ₹20,321,299
Average price/sq.ft: ₹18,421
Average predicted price: ₹20,400,476
Average predicted price/sq.ft: ₹18,477


In [42]:
# ✅ Conclusion

The Interactive Mumbai Locality Map has been successfully created.

The project now provides:

- Interactive Mumbai property map
- Property-level geographic visualization
- Locality-level visualization
- Locality intelligence popups
- Property price information
- Actual price per sq.ft
- Predicted price per sq.ft
- Property count by locality
- Average property price by locality
- Price category classification
- Interactive map layers
- Saved HTML maps
- Locality summary CSV

Generated files:

1. `mumbai_interactive_property_map.html`
2. `mumbai_locality_map.html`
3. `mumbai_price_per_sqft_map.html`
4. `interactive_locality_summary.csv`

The generated maps can later be integrated into the Flask dashboard.

SyntaxError: invalid syntax (1398150990.py, line 3)